<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture - KPI Aggregation Showcase

Demonstrates KPI aggregation on time-series extractors.  
Each KPI example uses `MRTSExtractor` on a **single entity** to keep things simple.  
The same `kpi_filter` parameter works with VegationTsExtractor and WeatherExtractor.

## Step 1: Initialisation

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
from earthdaily.agriculture.extractors.VTS_functions import MRTSExtractor

manager = WorkflowManager('prod')

## Step 2: Load entities and pick a test entity

In [ ]:
manager.load_seasonfields(
    sowing_date_gte='2025-07-01',
    crop_id='WINTER_OSR',
)

# Pick a test entity
row = manager.sfd_list.iloc[4].to_dict()
print(f"Test entity: {row['id']}")

## Step 3: Create extractor

Column mapping bridges platform field names to extractor canonical names.

In [ ]:
mrts = MRTSExtractor(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config=manager.config,
    workflow_ref=manager
)

column_mapping = {'crop': 'crop.id', 'start_date': 'sowingDate'}

## Step 4: KPI Examples

Each cell reconfigures the extractor with a different `kpi_filter` and runs on the same entity.  
All examples use `mode='raw'` and `historical_years=5` for comparison.

### Helper function

In [ ]:
def run_kpi(kpi_filter, label=None):
    """Configure MRTS with a kpi_filter, run on test entity, display result."""
    mrts.setup_mrts_parameters(
        start_date='2025-11-01',
        end_date='2026-02-28',
        vegetation_index='NDVI',
        mode='raw',
        historical_years=5,
        column_mapping=column_mapping,
        kpi_filter=kpi_filter,
    )
    result = mrts.process_single_entity_mrts(row=row)
    if result['error']:
        print(f"Error: {result['error']['message']}")
    else:
        df = result['data']
        kpi_cols = ['kpi_name', 'aggregation', 'current_value',
                    'current_num_records', 'historical_avg',
                    'historical_num_years', 'difference', 'percent_change']
        display(df[[c for c in kpi_cols if c in df.columns]])
    return result

### accumulation — Total NDVI over the season

Sums all raw NDVI values in the period. Useful to measure total vegetation productivity.

In [ ]:
run_kpi({
    'kpi_name': 'Season NDVI Total',
    'aggregation': 'accumulation',
})

### top_accumulation — Sum of the 30 highest values

Sorts values descending, takes the top N (threshold), and sums them.  
Focuses on peak performance days, filtering out noise and low-quality observations.

In [ ]:
run_kpi({
    'kpi_name': 'NDVI Top-30 Accumulation',
    'aggregation': 'top_accumulation',
    'threshold': 30,
})

### average — Mean NDVI

Average of all values in the period. Less sensitive to the number of observations than accumulation.

In [ ]:
run_kpi({
    'kpi_name': 'Average NDVI',
    'aggregation': 'average',
})

### max — Peak NDVI

Highest single value observed in the period. Indicates maximum vegetation vigor reached.

In [ ]:
run_kpi({
    'kpi_name': 'Peak NDVI',
    'aggregation': 'max',
})

### min — Minimum NDVI

Lowest value in the period. Useful for detecting stress events or bare soil exposure.

In [ ]:
run_kpi({
    'kpi_name': 'Min NDVI',
    'aggregation': 'min',
})

### std — NDVI variability

Standard deviation of values. High variability may indicate inconsistent growth or data quality issues.

In [ ]:
run_kpi({
    'kpi_name': 'NDVI Variability',
    'aggregation': 'std',
})

### count_gt — Days with NDVI above 0.7

Counts how many observations exceed the threshold. Measures duration of high vegetation vigor.

In [ ]:
run_kpi({
    'kpi_name': 'High Vigor Days (NDVI > 0.7)',
    'aggregation': 'count_gt',
    'threshold': 0.7,
})

### count_lt — Days with NDVI below 0.3

Counts observations below the threshold. Detects extended periods of low vegetation cover.

In [ ]:
run_kpi({
    'kpi_name': 'Low Cover Days (NDVI < 0.3)',
    'aggregation': 'count_lt',
    'threshold': 0.3,
})

### count_between — Days with NDVI in a range

Counts observations within a value band. Threshold is a `(min, max)` tuple.

In [ ]:
run_kpi({
    'kpi_name': 'Moderate NDVI Days (0.3-0.7)',
    'aggregation': 'count_between',
    'threshold': (0.3, 0.7),
})

### rolling_avg — Rolling mean over an integration period

Smooths the series with a rolling window of `window` consecutive records, then returns the mean of the smoothed series. `window` is required and counts records, not days — for sparse satellite series, 3–5 acquisitions; for dense daily weather, 7–14 days.

In [ ]:
run_kpi({
    'kpi_name': 'NDVI 3-Obs Rolling Avg',
    'aggregation': 'rolling_avg',
    'window': 3,
})

### rolling_avg_gt — Rolling-mean periods above a value

Counts how many points of the rolling-mean series exceed `threshold`. Captures *sustained* high vigor (multiple consecutive observations) instead of isolated peaks. `window` sets the integration period in number of consecutive records — 3–5 records works well for raw MRTS NDVI; tune larger for smoother series.

In [ ]:
run_kpi({
    'kpi_name': 'High NDVI Rolling Periods (>0.6)',
    'aggregation': 'rolling_avg_gt',
    'window': 3,
    'threshold': 0.6,
})

### rolling_avg_lt — Rolling-mean periods below a value

Counts how many points of the rolling-mean series are strictly below `threshold`. Detects sustained low-vigor stretches — stress windows, late dormancy, post-harvest exposure — while ignoring single-image dips caused by clouds or sensor noise.

In [ ]:
run_kpi({
    'kpi_name': 'Low NDVI Rolling Periods (<0.4)',
    'aggregation': 'rolling_avg_lt',
    'window': 3,
    'threshold': 0.4,
})

### percentile — Value at the Pth percentile

Returns the value at the Pth percentile of the period. `threshold` is the percentile rank (0–100). E.g. `threshold=90` returns the 90th-percentile NDVI — a robust upper-tail summary that's less sensitive to single peak observations than `max`.

In [ ]:
run_kpi({
    'kpi_name': 'NDVI 90th Percentile',
    'aggregation': 'percentile',
    'threshold': 90,
})

### percentile_gt — Days above the period's Pth percentile

Counts observations strictly above the Pth percentile of the same period. `threshold` is the percentile rank (0–100). Combined with `historical_years` it surfaces seasons whose upper tail is unusually populated compared to the historical baseline.

In [ ]:
run_kpi({
    'kpi_name': 'Days Above 90th Percentile',
    'aggregation': 'percentile_gt',
    'threshold': 90,
})

### percentile_lt — Days below the period's Pth percentile

Counts observations strictly below the Pth percentile of the same period. `threshold` is the percentile rank (0–100). Most useful in combination with `historical_years` to flag seasons whose lower tail is unusually concentrated.

In [ ]:
run_kpi({
    'kpi_name': 'Days Below 10th Percentile',
    'aggregation': 'percentile_lt',
    'threshold': 10,
})

### Crop-specific historical comparison — `historical_years` as a list

By default `historical_years` is an integer (N-year lookback) which compares the current season against the previous N years regardless of what crop was grown.

For **crop-specific comparison** you can pass a **list of years** (or a comma-separated string) so that only seasons where the same crop was planted are used as the historical baseline.

Typical workflow:
1. Run CropID with `mode='history'` to find which years a given crop was grown.
2. Feed the resulting year list into `historical_years` for VTS / MRTS / Weather extraction.

Accepted formats:
- `historical_years=[2021, 2023, 2025]` — explicit list of years
- `historical_years="2021,2023,2025"` — comma-separated string (e.g. from CropID output)
- `historical_years="ALL"` — use all available years

In [ ]:
# Suppose CropID told us WINTER_OSR was grown in 2021, 2023, and 2025.
# Pass those years as a list so the KPI historical baseline only uses
# seasons where the same crop was planted.

crop_years = [2021, 2023, 2025]   # from CropID historical_season output

mrts.setup_mrts_parameters(
    start_date='2025-11-01',
    end_date='2026-02-28',
    vegetation_index='NDVI',
    mode='raw',
    historical_years=crop_years,    # list → only these years in the baseline
    column_mapping=column_mapping,
    kpi_filter={
        'kpi_name': 'Crop-Specific Peak NDVI',
        'aggregation': 'max',
    },
)

result = mrts.process_single_entity_mrts(row=row)
df = result['data']
kpi_cols = ['kpi_name', 'aggregation', 'current_value',
            'current_num_records', 'historical_avg',
            'historical_num_years', 'difference', 'percent_change']
display(df[[c for c in kpi_cols if c in df.columns]])

In [ ]:
# Same result using a comma-separated string (e.g. directly from CropID output)

mrts.setup_mrts_parameters(
    start_date='2025-11-01',
    end_date='2026-02-28',
    vegetation_index='NDVI',
    mode='raw',
    historical_years="2021,2023,2025",  # string → parsed to [2021, 2023, 2025]
    column_mapping=column_mapping,
    kpi_filter={
        'kpi_name': 'Crop-Specific Season Accumulation',
        'aggregation': 'accumulation',
    },
)

result = mrts.process_single_entity_mrts(row=row)
df = result['data']
display(df[[c for c in kpi_cols if c in df.columns]])

## Step 5: Standalone KPI on any DataFrame

`filter_timeseries_kpi` can be called directly on any DataFrame with date and value columns.  
First extract the raw time series, then compute KPIs from the result.

In [ ]:
# Extract time series in full mode (raw + smoothed) — used by the standalone KPI loops
# below and the interactive explorer in Step 6.
mrts.setup_mrts_parameters(
    start_date='2025-11-01',
    end_date='2026-02-28',
    vegetation_index='NDVI',
    mode='full',
    historical_years=5,
    column_mapping=column_mapping,
    kpi_filter=None,
)

raw_result = mrts.process_single_entity_mrts(row=row)
ts_df = raw_result['data']
print(f"Time series shape: {ts_df.shape}")
print(f"Columns: {ts_df.columns.tolist()}")
display(ts_df.head())

In [ ]:
from earthdaily.agriculture.core.api_utils import filter_timeseries_kpi, format_kpi_results

# Compute multiple KPIs from the same time series
kpis = [
    ('Season Total',    'accumulation', None),
    ('Top-20 Sum',      'top_accumulation', 20),
    ('Average',         'average', None),
    ('Peak',            'max', None),
    ('High Vigor Days', 'count_gt', 0.7),
]

for name, agg, thresh in kpis:
    result = filter_timeseries_kpi(
        timeseries_df=ts_df,
        start_date='2025-11-01',
        end_date='2026-02-28',
        kpi_name=name,
        aggregation=agg,
        threshold=thresh,
        years='ALL',
        date_column='date',
        value_column='raw_value',
    )
    display(format_kpi_results(result))

In [ ]:
# Standalone usage of the rolling and percentile aggregations.
# Uses dict configs because some ops need both `window` and `threshold`.

new_kpis = [
    {'kpi_name': '3-Obs Rolling Avg',         'aggregation': 'rolling_avg',    'window': 3},
    {'kpi_name': 'Rolling Periods > 0.6',     'aggregation': 'rolling_avg_gt', 'window': 3, 'threshold': 0.6},
    {'kpi_name': 'Rolling Periods < 0.4',     'aggregation': 'rolling_avg_lt', 'window': 3, 'threshold': 0.4},
    {'kpi_name': '90th Percentile NDVI',      'aggregation': 'percentile',     'threshold': 90},
    {'kpi_name': 'Days Above 90th Pctile',    'aggregation': 'percentile_gt',  'threshold': 90},
    {'kpi_name': 'Days Below 10th Pctile',    'aggregation': 'percentile_lt',  'threshold': 10},
]

for cfg in new_kpis:
    result = filter_timeseries_kpi(
        timeseries_df=ts_df,
        start_date='2025-11-01',
        end_date='2026-02-28',
        years='ALL',
        date_column='date',
        value_column='raw_value',
        **cfg,
    )
    display(format_kpi_results(result))

## Step 6: Interactive KPI Explorer

Quick interactive playground for demos and ad‑hoc series analysis. **Data fetching is decoupled from analysis** — the explorer runs entirely client‑side on the `ts_df` extracted earlier (Step 5), so changing parameters never re‑hits the API.

Pick an aggregation in the dropdown, tune its parameters with the sliders, and watch:
- the **KPI value + historical comparison** update live in the banner above the chart
- the **chart overlays** update to reflect what's being measured:
  - count / rolling / percentile *gt/lt/between* ops → threshold lines + highlighted event markers
  - `rolling_avg*` → rolling‑mean line in green
  - `accumulation` → cumulative sum on a secondary y‑axis
  - `top_accumulation` → top‑N points highlighted, cutoff line shown
  - `average` / `max` / `min` / `percentile` → horizontal reference line
  - `std` → mean line + ±1σ band

Requires `ipywidgets` and `anywidget` (Plotly ≥ 6 backs `FigureWidget` on `anywidget`). If either is missing the cell raises a clear `ImportError` — install with `pip install ipywidgets anywidget`. Both are pinned in `requirements.txt` and the `[jupyter]` extra in `pyproject.toml`.

In [ ]:
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("This cell needs ipywidgets — install with `pip install ipywidgets`.") from exc
try:
    import anywidget  # noqa: F401  — required by Plotly>=6 FigureWidget
except ImportError as exc:
    raise ImportError(
        "This cell needs anywidget (Plotly>=6 backs FigureWidget on it) — install with `pip install anywidget`."
    ) from exc

from datetime import datetime
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

from earthdaily.agriculture.core.api_utils import filter_timeseries_kpi


def kpi_explorer(ts_df, date_col, value_col, start_date, end_date, years='ALL', export_dir=None):
    """Interactive KPI explorer. Data fetching is done upstream — pass a pre-extracted DataFrame.

    Pick an aggregation, tune its parameters with sliders, and see the resulting events /
    reference series overlaid on the time series. Click 'Export CSV' to dump the current
    chart's data (observations + computed series + event mask) to a timestamped file.

    Args:
        ts_df: DataFrame containing the time series.
        date_col: name of the date column.
        value_col: name of the value column.
        start_date / end_date: KPI period bounds (YYYY-MM-DD).
        years: historical comparison spec passed through to filter_timeseries_kpi.
        export_dir: directory for CSV exports. Defaults to current working directory.
    """
    df = ts_df[[date_col, value_col]].dropna().copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    # Pre-compute ISO date strings for Plotly. Sending datetime64[ns] directly through
    # anywidget can serialize as epoch nanoseconds (huge ints) and confuse the date axis,
    # collapsing all points onto a single tick.
    date_str = df[date_col].dt.strftime('%Y-%m-%d')
    x_min_str = date_str.iloc[0]
    x_max_str = date_str.iloc[-1]

    AGGREGATIONS = [
        'accumulation', 'top_accumulation', 'average', 'max', 'min', 'std',
        'count_gt', 'count_lt', 'count_between',
        'rolling_avg', 'rolling_avg_gt', 'rolling_avg_lt',
        'percentile', 'percentile_gt', 'percentile_lt',
    ]

    # Slider ranges adapt to the loaded series
    v_min = float(df[value_col].min())
    v_max = float(df[value_col].max())
    v_range = max(v_max - v_min, 1e-6)
    v_step = max(round(v_range / 100, 4), 0.001)
    n_records = len(df)

    sty = {'description_width': 'initial'}
    agg_w = widgets.Dropdown(options=AGGREGATIONS, value='count_gt',
                             description='Aggregation:', style=sty)
    threshold_w = widgets.FloatSlider(value=v_min + 0.7 * v_range, min=v_min, max=v_max,
                                      step=v_step, description='Threshold:',
                                      continuous_update=False, style=sty,
                                      layout=widgets.Layout(width='420px'))
    threshold_min_w = widgets.FloatSlider(value=v_min + 0.3 * v_range, min=v_min, max=v_max,
                                          step=v_step, description='Min:',
                                          continuous_update=False, style=sty,
                                          layout=widgets.Layout(width='420px'))
    threshold_max_w = widgets.FloatSlider(value=v_min + 0.7 * v_range, min=v_min, max=v_max,
                                          step=v_step, description='Max:',
                                          continuous_update=False, style=sty,
                                          layout=widgets.Layout(width='420px'))
    top_n_w = widgets.IntSlider(value=min(10, n_records), min=1, max=max(n_records, 1),
                                step=1, description='Top N:', continuous_update=False, style=sty,
                                layout=widgets.Layout(width='420px'))
    percentile_w = widgets.FloatSlider(value=90, min=0, max=100, step=1,
                                       description='Percentile (0–100):',
                                       continuous_update=False, style=sty,
                                       layout=widgets.Layout(width='420px'))
    window_w = widgets.IntSlider(value=3, min=1, max=max(min(30, n_records), 1), step=1,
                                 description='Window (records):',
                                 continuous_update=False, style=sty,
                                 layout=widgets.Layout(width='420px'))
    export_btn = widgets.Button(description='Export CSV', icon='download',
                                button_style='primary', tooltip='Save current chart data to CSV')

    info = widgets.HTML(value='')
    export_status = widgets.HTML(value='')

    NEEDS = {
        'accumulation':     set(),
        'top_accumulation': {'top_n'},
        'average':          set(),
        'max':              set(),
        'min':              set(),
        'std':              set(),
        'count_gt':         {'threshold'},
        'count_lt':         {'threshold'},
        'count_between':    {'threshold_min', 'threshold_max'},
        'rolling_avg':      {'window'},
        'rolling_avg_gt':   {'threshold', 'window'},
        'rolling_avg_lt':   {'threshold', 'window'},
        'percentile':       {'percentile'},
        'percentile_gt':    {'percentile'},
        'percentile_lt':    {'percentile'},
    }
    WIDGETS_BY_KEY = {
        'threshold': threshold_w, 'threshold_min': threshold_min_w,
        'threshold_max': threshold_max_w, 'top_n': top_n_w,
        'percentile': percentile_w, 'window': window_w,
    }

    def sync_visibility(*_):
        needs = NEEDS[agg_w.value]
        for key, w in WIDGETS_BY_KEY.items():
            w.layout.display = '' if key in needs else 'none'

    # State captured by update() and consumed by the export handler so the CSV reflects
    # exactly what the user sees on screen.
    state = {
        'rolling': None,        # pd.Series aligned to df, or None
        'cumsum': None,         # pd.Series aligned to df, or None
        'event_mask': None,     # bool Series aligned to df, or None
        'kpi_result': None,     # last kpi result dict
        'kpi_kwargs': None,     # the kwargs used to compute it
    }

    # Pre-allocate traces; index order is fixed and reused on update
    fig = go.FigureWidget()
    fig.add_scatter(x=date_str, y=df[value_col], mode='markers',
                    name='observations', marker=dict(size=6, color='#444'))
    fig.add_scatter(x=[], y=[], mode='markers', name='events',
                    marker=dict(size=11, color='red', symbol='x'))
    fig.add_scatter(x=[], y=[], mode='lines', name='rolling mean',
                    line=dict(color='#2ca02c', width=2))
    fig.add_scatter(x=[x_min_str, x_max_str], y=[None, None],
                    mode='lines', name='reference',
                    line=dict(color='#ff7f0e', dash='dash'))
    fig.add_scatter(x=[x_min_str, x_max_str], y=[None, None],
                    mode='lines', name='reference 2',
                    line=dict(color='#ff7f0e', dash='dash'), visible=False)
    fig.add_scatter(x=[], y=[], mode='lines', name='+1σ',
                    line=dict(color='#9467bd', dash='dot'), visible=False)
    fig.add_scatter(x=[], y=[], mode='lines', name='–1σ',
                    line=dict(color='#9467bd', dash='dot'), visible=False)
    fig.add_scatter(x=[], y=[], mode='lines', name='cumulative sum',
                    line=dict(color='#1f77b4', width=2), yaxis='y2', visible=False)
    BASE, EVENTS, ROLLING, REF, REF2, BAND_HI, BAND_LO, CUMSUM = range(8)

    fig.update_layout(
        title=f'{value_col} — KPI Explorer',
        # Force date axis with explicit format so ticks render as dates, not numbers.
        xaxis=dict(title='date', type='date', tickformat='%Y-%m-%d', hoverformat='%Y-%m-%d'),
        yaxis=dict(title=value_col),
        yaxis2=dict(title='cumulative', overlaying='y', side='right', showgrid=False),
        height=460, hovermode='x unified', template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )

    def build_kpi_kwargs():
        agg = agg_w.value
        kw = {'aggregation': agg}
        if 'threshold' in NEEDS[agg]:
            kw['threshold'] = threshold_w.value
        if 'top_n' in NEEDS[agg]:
            kw['threshold'] = int(top_n_w.value)
        if 'threshold_min' in NEEDS[agg]:
            kw['threshold'] = (threshold_min_w.value, threshold_max_w.value)
        if 'percentile' in NEEDS[agg]:
            kw['threshold'] = percentile_w.value
        if 'window' in NEEDS[agg]:
            kw['window'] = int(window_w.value)
        return kw

    def update(*_):
        try:
            kwargs = build_kpi_kwargs()
            result = filter_timeseries_kpi(
                timeseries_df=df, start_date=start_date, end_date=end_date,
                kpi_name=f'{value_col} {agg_w.value}',
                years=years, date_column=date_col, value_column=value_col,
                **kwargs,
            )
        except Exception as e:
            info.value = f'<span style="color:red">Error: {e}</span>'
            return

        # Reset state for this update
        state['rolling'] = None
        state['cumsum'] = None
        state['event_mask'] = None
        state['kpi_result'] = result
        state['kpi_kwargs'] = kwargs

        agg = agg_w.value

        with fig.batch_update():
            # Reset overlays to a clean state
            for idx in (EVENTS, ROLLING, BAND_HI, BAND_LO, CUMSUM):
                fig.data[idx].x = []
                fig.data[idx].y = []
            fig.data[REF].y = [None, None]
            fig.data[REF2].y = [None, None]
            fig.data[REF2].visible = False
            fig.data[BAND_HI].visible = False
            fig.data[BAND_LO].visible = False
            fig.data[CUMSUM].visible = False

            mask = None
            if agg == 'accumulation':
                cum = df[value_col].cumsum()
                state['cumsum'] = cum
                fig.data[CUMSUM].x = date_str
                fig.data[CUMSUM].y = cum
                fig.data[CUMSUM].visible = True
            elif agg == 'top_accumulation':
                n = int(top_n_w.value)
                top_idx = df[value_col].nlargest(n).index
                mask = df.index.isin(top_idx)
                fig.data[EVENTS].x = date_str[mask]
                fig.data[EVENTS].y = df.loc[mask, value_col]
                if mask.any():
                    cutoff = float(df.loc[mask, value_col].min())
                    fig.data[REF].y = [cutoff, cutoff]
            elif agg == 'average':
                m = float(df[value_col].mean())
                fig.data[REF].y = [m, m]
            elif agg == 'max':
                m = float(df[value_col].max())
                fig.data[REF].y = [m, m]
            elif agg == 'min':
                m = float(df[value_col].min())
                fig.data[REF].y = [m, m]
            elif agg == 'std':
                m = float(df[value_col].mean())
                s = float(df[value_col].std())
                fig.data[REF].y = [m, m]
                fig.data[BAND_HI].x = [x_min_str, x_max_str]
                fig.data[BAND_HI].y = [m + s, m + s]
                fig.data[BAND_HI].visible = True
                fig.data[BAND_LO].x = [x_min_str, x_max_str]
                fig.data[BAND_LO].y = [m - s, m - s]
                fig.data[BAND_LO].visible = True
            elif agg == 'count_gt':
                t = threshold_w.value
                mask = (df[value_col] > t).values
                fig.data[REF].y = [t, t]
                fig.data[EVENTS].x = date_str[mask]
                fig.data[EVENTS].y = df.loc[mask, value_col]
            elif agg == 'count_lt':
                t = threshold_w.value
                mask = (df[value_col] < t).values
                fig.data[REF].y = [t, t]
                fig.data[EVENTS].x = date_str[mask]
                fig.data[EVENTS].y = df.loc[mask, value_col]
            elif agg == 'count_between':
                lo, hi = threshold_min_w.value, threshold_max_w.value
                mask = ((df[value_col] >= lo) & (df[value_col] <= hi)).values
                fig.data[REF].y = [lo, lo]
                fig.data[REF2].y = [hi, hi]
                fig.data[REF2].visible = True
                fig.data[EVENTS].x = date_str[mask]
                fig.data[EVENTS].y = df.loc[mask, value_col]
            elif agg in ('rolling_avg', 'rolling_avg_gt', 'rolling_avg_lt'):
                w = int(window_w.value)
                roll = df[value_col].rolling(window=w, min_periods=w).mean()
                state['rolling'] = roll
                fig.data[ROLLING].x = date_str
                fig.data[ROLLING].y = roll
                if agg == 'rolling_avg_gt':
                    t = threshold_w.value
                    mask = (roll > t).fillna(False).values
                    fig.data[REF].y = [t, t]
                    fig.data[EVENTS].x = date_str[mask]
                    fig.data[EVENTS].y = roll[mask]
                elif agg == 'rolling_avg_lt':
                    t = threshold_w.value
                    mask = (roll < t).fillna(False).values
                    fig.data[REF].y = [t, t]
                    fig.data[EVENTS].x = date_str[mask]
                    fig.data[EVENTS].y = roll[mask]
            elif agg in ('percentile', 'percentile_gt', 'percentile_lt'):
                p = percentile_w.value
                cutoff = float(df[value_col].quantile(p / 100))
                fig.data[REF].y = [cutoff, cutoff]
                if agg == 'percentile_gt':
                    mask = (df[value_col] > cutoff).values
                    fig.data[EVENTS].x = date_str[mask]
                    fig.data[EVENTS].y = df.loc[mask, value_col]
                elif agg == 'percentile_lt':
                    mask = (df[value_col] < cutoff).values
                    fig.data[EVENTS].x = date_str[mask]
                    fig.data[EVENTS].y = df.loc[mask, value_col]

            if mask is not None:
                state['event_mask'] = pd.Series(mask, index=df.index)

        cv = result['current_period']['value']
        nr = result['current_period']['num_records']
        ha = result['historical_avg']['value']
        ny = result['historical_avg']['num_years']
        pc = result['comparison']['percent_change']
        pc_str = f"{pc:+.2f}%" if pc is not None else '—'
        info.value = (
            f"<div style='font-size:14px'>"
            f"<b>{result['kpi_name']}</b> — "
            f"current: <b>{cv}</b> ({nr} records) &nbsp;|&nbsp; "
            f"historical avg: <b>{ha}</b> ({ny} years) &nbsp;|&nbsp; "
            f"% change: <b>{pc_str}</b>"
            f"</div>"
        )
        # Clear stale export message on every update
        export_status.value = ''

    def on_export(_):
        out = df[[date_col, value_col]].copy()
        out[date_col] = out[date_col].dt.strftime('%Y-%m-%d')
        if state['rolling'] is not None:
            out['rolling_mean'] = state['rolling'].values
        if state['cumsum'] is not None:
            out['cumulative_sum'] = state['cumsum'].values
        if state['event_mask'] is not None:
            out['is_event'] = state['event_mask'].values

        # Attach KPI summary as trailing metadata rows so users get the headline
        # numbers without opening a second file.
        result = state['kpi_result'] or {}
        kw = state['kpi_kwargs'] or {}
        meta_rows = pd.DataFrame([
            {date_col: '#aggregation', value_col: kw.get('aggregation')},
            {date_col: '#threshold',   value_col: str(kw.get('threshold', ''))},
            {date_col: '#window',      value_col: str(kw.get('window', ''))},
            {date_col: '#current',     value_col: result.get('current_period', {}).get('value')},
            {date_col: '#historical_avg', value_col: result.get('historical_avg', {}).get('value')},
            {date_col: '#percent_change', value_col: result.get('comparison', {}).get('percent_change')},
        ])
        # Align metadata columns with the data frame
        for col in out.columns:
            if col not in meta_rows.columns:
                meta_rows[col] = None
        out = pd.concat([out, meta_rows[out.columns]], ignore_index=True)

        out_dir = Path(export_dir) if export_dir else Path.cwd()
        out_dir.mkdir(parents=True, exist_ok=True)
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        agg = (state['kpi_kwargs'] or {}).get('aggregation', 'kpi')
        path = out_dir / f'kpi_explorer_{value_col}_{agg}_{ts}.csv'
        out.to_csv(path, index=False)
        export_status.value = (
            f"<span style='color:green'>✓ Exported {len(out) - len(meta_rows)} rows "
            f"(+ {len(meta_rows)} metadata) → <code>{path}</code></span>"
        )

    for w in (agg_w, threshold_w, threshold_min_w, threshold_max_w,
              top_n_w, percentile_w, window_w):
        w.observe(update, names='value')
    agg_w.observe(sync_visibility, names='value')
    export_btn.on_click(on_export)

    sync_visibility()
    update()

    controls = widgets.VBox([
        agg_w,
        widgets.HBox([threshold_w, threshold_min_w, threshold_max_w]),
        widgets.HBox([top_n_w, percentile_w, window_w]),
        widgets.HBox([export_btn, export_status]),
    ])
    return widgets.VBox([controls, info, fig])


# Launch the explorer on the smoothed time series extracted in Step 5 (mode='full').
# Slider changes never re-hit the API. Click 'Export CSV' to dump the current view.
kpi_explorer(
    ts_df=ts_df,
    date_col='date',
    value_col='smoothed_value',
    start_date='2025-11-01',
    end_date='2026-02-28',
    years='ALL',
)